In [0]:
# Manipulação e estruturação de dados
import pandas as pd
import numpy as np

# Visualização de dados
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning e Métricas (caso vá avançar para modelagem preditiva)
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

Vendo os dados de forma geral e limitando a 30 registros

In [0]:
%sql
select * from workspace.default.bank_churn_dataset limit 30

Descrevendo os dados de modo geral

In [0]:

%sql
describe workspace.default.bank_churn_dataset;

Passo 1: Criação da Tabela Limpa e Padronizada (SQL)  - PT BR

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.bank_churn_dataset_pt AS
SELECT 
    id                     AS id_cliente,
    full_name              AS nome_completo,
    credit_sco             AS score_credito,
    gender                 AS genero,
    age                    AS idade,
    occupation             AS profissao,
    balance                AS saldo,
    monthly_ir             AS renda_mensal,
    address                AS endereco,
    origin_province        AS provincia_origem,
    tenure_ye              AS anos_relacionamento,
    married                AS casado,
    nums_card              AS num_cartoes,
    nums_service           AS num_servicos,
    active_member          AS membro_ativo,
    last_active_date       AS data_ultima_atividade,
    last_transaction_month AS ultimo_mes_transacao,
    created_date           AS data_criacao,
   CASE WHEN exit = TRUE THEN 1 ELSE 0 END AS churn,
    customer_segment       AS segmento_cliente,
    engagement_score       AS score_engajamento,
    loyalty_level          AS nivel_fidelidade,
    digital_behavior       AS comportamento_digital,
    risk_score             AS score_risco,
    risk_segment           AS segmento_risco,
    cluster_group          AS grupo_cluster
FROM workspace.default.bank_churn_dataset;

Iniciando EDA com perguntas básicas para compreeder a base

Qual é a Taxa de Churn Geral da Base?

In [0]:
%sql
SELECT 
    COUNT(*) AS total_clientes,
    SUM(CASE WHEN churn = 1 THEN 1 ELSE 0 END) AS total_churn,
    SUM(CASE WHEN churn = 0 THEN 1 ELSE 0 END) AS total_ativos,
    ROUND(SUM(CASE WHEN churn = 1 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_churn_pct,
    ROUND(SUM(CASE WHEN churn = 0 THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS taxa_ativos_pct
FROM workspace.default.bank_churn_dataset_pt;

Observamos um total de 80000 clientes
clientes que deram churn foram = 14400 = 18% da base
Clientes ativos = 65500 

B. Perfil Demográfico: Idade × Churn

In [0]:
%sql
SELECT 
    churn,
    COUNT(*) AS qtd_clientes,
    ROUND(AVG(idade), 1) AS media_idade,
    ROUND(MIN(idade), 0) AS idade_minima,
    ROUND(MAX(idade), 0) AS idade_maxima,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY churn;

Saúde Financeira: Saldo e Renda Mensal × Churn

In [0]:
%sql
select
    churn,
    round(avg(saldo),2) as media_saldo,
    round(avg(renda_mensal),2) as media_renda_mensal,
    round(avg(score_credito),1) as media_score_credito,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
from workspace.default.bank_churn_dataset_pt
group by churn

In [0]:
%sql
SELECT 
    membro_ativo,
    churn,
    COUNT(*) AS total_clientes,
    (COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()) AS percentual_total
FROM workspace.default.bank_churn_dataset_pt
GROUP BY membro_ativo, churn
ORDER BY membro_ativo, churn;